# Comparison: the three DAMICORE experiments

This notebook reads the standardized result packs only. It does not query the
database and does not execute DAMICORE.


In [ ]:
from pathlib import Path
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").exists():
    raise RuntimeError("Run this notebook from the repository or one of its subdirectories.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hypotheses.violence_against_women.scripts.experiment_common import (
    artifact_paths,
    load_artifact_manifest,
    load_category_map,
)
load_dotenv(PROJECT_ROOT / ".env")
CATEGORY_SET_VERSION = os.getenv("DAMICORE_CATEGORY_SET_VERSION", "v2_30")
PATHS = artifact_paths(CATEGORY_SET_VERSION)
COMMON_WORK_ROOT = PATHS.common
NORMALIZED_WORK_ROOT = PATHS.normalized
CASE_FULL_WORK_ROOT = PATHS.case_full
CASE_BALANCED_WORK_ROOT = PATHS.case_balanced
RESULTS_ROOT = PATHS.results


from itertools import combinations

import matplotlib.pyplot as plt
import numpy as np

from hypotheses.violence_against_women.scripts.experiment_common import (
    BIAS_WARNING_THRESHOLD,
    NCD_COLOR_VMAX,
    category_order_from_manifest,
    compare_distance_matrices,
    plot_distance_agreement,
    same_cluster_pairs,
    wrapped_label,
    write_json,
)

manifest = load_artifact_manifest(
    COMMON_WORK_ROOT / "artifact-manifest.json",
    category_set_version=CATEGORY_SET_VERSION,
)
category_order = category_order_from_manifest(manifest)
category_map = load_category_map(COMMON_WORK_ROOT / "category-map.csv")
experiment_names = ["case_full", "case_balanced", "normalized_categories"]
result_dirs = {name: RESULTS_ROOT / name for name in experiment_names}
for name, directory in result_dirs.items():
    if not (directory / "distance-matrix.csv").exists():
        raise FileNotFoundError(f"Missing standardized result for {name}: {directory}")


## Load matrices, memberships and support


In [ ]:
distance_matrices = {
    name: pd.read_csv(
        directory / "distance-matrix.csv", index_col=0
    ).loc[category_order, category_order]
    for name, directory in result_dirs.items()
}
memberships = {
    name: pd.read_csv(directory / "clusters.csv")
    for name, directory in result_dirs.items()
}
supports = {
    name: pd.read_csv(directory / "support.csv")
    for name, directory in result_dirs.items()
}
diagnostics = {
    name: pd.read_csv(directory / "bias-diagnostics.csv")
    for name, directory in result_dirs.items()
}
assert all(np.allclose(np.diag(matrix.to_numpy()), 0.0) for matrix in distance_matrices.values())


## Distance agreement and common-scale heatmaps


In [ ]:
comparison_dir = RESULTS_ROOT / "comparison"
comparison_dir.mkdir(parents=True, exist_ok=True)
distance_comparison = compare_distance_matrices(distance_matrices, category_order)
distance_comparison.to_csv(comparison_dir / "distance-agreement.csv", index=False)

agreement_names = list(distance_matrices)
agreement_values = np.eye(len(agreement_names))
for row in distance_comparison.itertuples(index=False):
    left = agreement_names.index(row.left)
    right = agreement_names.index(row.right)
    agreement_values[left, right] = row.spearman_correlation
    agreement_values[right, left] = row.spearman_correlation
agreement = pd.DataFrame(agreement_values, index=agreement_names, columns=agreement_names)
agreement.to_csv(comparison_dir / "distance-agreement-matrix.csv")
plot_distance_agreement(agreement, comparison_dir / "distance-agreement.png")

heatmap_labels = [wrapped_label(category, 20) for category in category_order]
figure, axes = plt.subplots(1, 3, figsize=(21, 8), constrained_layout=True)
for axis, (name, matrix) in zip(axes, distance_matrices.items()):
    image = axis.imshow(matrix.to_numpy(), cmap="Blues", vmin=0, vmax=NCD_COLOR_VMAX)
    labels = heatmap_labels
    axis.set_xticks(np.arange(len(labels)))
    axis.set_xticklabels(labels, rotation=90, fontsize=6)
    axis.set_yticks(np.arange(len(labels)))
    axis.set_yticklabels(labels, fontsize=6)
    axis.set_title(name)
figure.colorbar(image, ax=axes.tolist(), label="NCD", shrink=0.82)
figure.savefig(comparison_dir / "ncd-heatmaps-comparison.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.close(figure)


## Label-invariant cluster agreement


In [ ]:
cluster_rows = []
pair_sets = {
    name: same_cluster_pairs(memberships[name], category_order)
    for name in experiment_names
}
all_pairs = set(combinations(category_order, 2))
for left_name, right_name in combinations(experiment_names, 2):
    left_pairs = pair_sets[left_name]
    right_pairs = pair_sets[right_name]
    cluster_rows.append({
        "left": left_name,
        "right": right_name,
        "pairwise_cluster_agreement": len(all_pairs - (left_pairs ^ right_pairs))
        / len(all_pairs),
    })
cluster_agreement = pd.DataFrame(cluster_rows)
cluster_agreement.to_csv(comparison_dir / "cluster-agreement.csv", index=False)
display(distance_comparison)
display(cluster_agreement)


## Support and size diagnostics


In [ ]:
diagnostic_rows = []
for name, table in diagnostics.items():
    for row in table.itertuples(index=False):
        diagnostic_rows.append({
            "experiment": name,
            "diagnostic": row.diagnostic,
            "spearman_correlation": row.spearman_correlation,
            "absolute_correlation": abs(row.spearman_correlation),
        })
diagnostics_comparison = pd.DataFrame(diagnostic_rows)
diagnostics_comparison.to_csv(
    comparison_dir / "bias-diagnostics-comparison.csv", index=False
)
figure, axis = plt.subplots(figsize=(12, 6))
for name, table in diagnostics.items():
    axis.plot(
        table["diagnostic"],
        table["spearman_correlation"].abs(),
        marker="o",
        label=name,
    )
axis.axhline(
    BIAS_WARNING_THRESHOLD,
    color="#C44536",
    linestyle="--",
    linewidth=1.2,
    label=f"screening threshold ({BIAS_WARNING_THRESHOLD:.2f})",
)
axis.set_ylim(0, 1)
axis.set_ylabel("Absolute Spearman correlation")
axis.set_title("Support and document-size diagnostics", loc="left", weight="bold")
axis.tick_params(axis="x", rotation=18)
axis.grid(axis="y", color="#E2E7E9", linewidth=0.7)
axis.set_axisbelow(True)
axis.legend(frameon=False)
figure.tight_layout()
figure.savefig(
    comparison_dir / "bias-diagnostics-comparison.png",
    dpi=180,
    bbox_inches="tight",
    facecolor="white",
)
plt.close(figure)

balanced_stability_path = RESULTS_ROOT / "case_balanced" / "replicate-stability.csv"
if balanced_stability_path.exists():
    balanced_stability = pd.read_csv(balanced_stability_path)
    balanced_stability.to_csv(
        comparison_dir / "balanced-replica-stability.csv", index=False
    )
else:
    balanced_stability = pd.DataFrame()


## Common support comparison


In [ ]:
figure, axis = plt.subplots(figsize=(14, 7))
positions = np.arange(len(category_order))
width = 0.25
for index, name in enumerate(experiment_names):
    table = supports[name].set_index("category").loc[category_order]
    axis.bar(
        positions + (index - 1) * width,
        table["support"],
        width,
        label=name,
    )
axis.set_yscale("log")
axis.set_xticks(positions)
axis.set_xticklabels([wrapped_label(category, 22) for category in category_order], rotation=90, fontsize=7)
axis.set_ylabel("Support (log scale)")
axis.set_title("Support across the three experiments", loc="left", weight="bold")
axis.legend(frameon=False)
axis.grid(axis="y", color="#E2E7E9", linewidth=0.7)
axis.set_axisbelow(True)
figure.tight_layout()
figure.savefig(comparison_dir / "support-comparison.png", dpi=180, bbox_inches="tight", facecolor="white")
plt.close(figure)

summary = distance_comparison.copy()
summary["comparison_type"] = "distance"
cluster_summary = cluster_agreement.copy()
cluster_summary["comparison_type"] = "cluster"
pd.concat([summary, cluster_summary], ignore_index=True, sort=False).to_csv(
    comparison_dir / "comparison-summary.csv", index=False
)
write_json(
    comparison_dir / "comparison-manifest.json",
    {
        "experiments": experiment_names,
        "category_set_version": manifest["category_set_version"],
        "category_definition_hash": manifest["category_definition_hash"],
        "category_count": manifest["category_count"],
        "dimension_count": manifest["dimension_count"],
        "category_order": category_order,
        "ncd_color_range": [0.0, NCD_COLOR_VMAX],
        "source_artifact_manifest": str(COMMON_WORK_ROOT / "artifact-manifest.json"),
    },
)
